# Причинное внимание и блок трансформера руками

Мы построим Q/K/V из обычных списков чисел, проверим одну строку внимания, затем соберём две головы, остаточные пути, нормализацию и позиционную MLP. Главный инвариант: изменение будущего токена не меняет ни один прошлый выход.

In [ ]:
import math
import platform

print('python:', platform.python_version())

def dot(left, right):
    return sum(a * b for a, b in zip(left, right))

def matvec(matrix, vector):
    return [dot(row, vector) for row in matrix]

def add(left, right):
    return [a + b for a, b in zip(left, right)]

def stable_softmax(values):
    maximum = max(values)
    weights = [math.exp(value - maximum) for value in values]
    total = sum(weights)
    return [weight / total for weight in weights]

def weighted_sum(weights, vectors):
    return [sum(weight * vector[column] for weight, vector in zip(weights, vectors))
            for column in range(len(vectors[0]))]

def assert_close(actual, expected, tolerance=1e-12):
    assert len(actual) == len(expected)
    assert all(abs(a - b) <= tolerance for a, b in zip(actual, expected)), (actual, expected)

states = [
    [1.0, 0.0, 0.5, -0.5],
    [0.0, 1.0, -0.5, 0.25],
    [0.5, 0.5, 1.0, 0.0],
    [-0.5, 1.0, 0.25, 0.75],
]
W_Q = [[1.0, 0.2, 0.0, 0.0], [0.0, 0.8, -0.1, 0.0],
       [0.1, 0.0, 0.9, 0.0], [0.0, -0.2, 0.0, 0.7]]
W_K = [[0.9, 0.0, 0.1, 0.0], [0.1, 1.0, 0.0, 0.0],
       [0.0, 0.0, 0.8, 0.2], [0.0, -0.1, 0.1, 0.9]]
W_V = [[1.0, 0.0, -0.2, 0.0], [0.0, 0.5, 0.5, 0.0],
       [0.2, -0.1, 0.9, 0.0], [0.0, 0.2, 0.0, 0.8]]
W_O = [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0],
       [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]]

## 1. Одна строка внимания

Запрос позиции t сравнивается только с ключами 0…t. На первой позиции доступен один ключ, поэтому softmax обязан вернуть [1], а выход — точно первый value.

In [ ]:
queries = [matvec(W_Q, state) for state in states]
keys = [matvec(W_K, state) for state in states]
values = [matvec(W_V, state) for state in states]

def one_attention_row(position):
    scores = [dot(queries[position], keys[source]) / math.sqrt(len(queries[position]))
              for source in range(position + 1)]
    probabilities = stable_softmax(scores)
    return scores, probabilities, weighted_sum(probabilities, values[:position + 1])

scores0, probabilities0, output0 = one_attention_row(0)
assert probabilities0 == [1.0]
assert_close(output0, values[0])

scores2, probabilities2, output2 = one_attention_row(2)
assert abs(sum(probabilities2) - 1.0) < 1e-12
print('scores t=2:', [round(value, 6) for value in scores2])
print('P t=2:     ', [round(value, 6) for value in probabilities2])
print('output:    ', [round(value, 6) for value in output2])

## 2. Масштабирование оценок внимания

В этом опыте масштабирование уменьшает разброс логитов и делает распределение менее насыщенным. Оно не меняет разрешённые позиции и не является маской.

In [ ]:
def entropy(probabilities):
    return -sum(p * math.log(p) for p in probabilities if p > 0.0)

wide_query = [4.0, -3.0, 2.0, 1.0]
wide_keys = [[3.0, -2.0, 1.0, 0.0], [-2.0, 3.0, 0.0, 1.0], [1.0, 1.0, -2.0, 2.0]]
unscaled = stable_softmax([dot(wide_query, key) for key in wide_keys])
scaled = stable_softmax([dot(wide_query, key) / math.sqrt(4) for key in wide_keys])
assert entropy(scaled) > entropy(unscaled)
print('entropy without scale:', round(entropy(unscaled), 6))
print('entropy with scale:   ', round(entropy(scaled), 6))

## 3. Две головы и полный блок

Ширина d=4 делится на H=2 головы по Dh=2 координаты. Каждая голова строит собственную строку вероятностей; их выходы конкатенируются, проходят W_O, остаточный путь и RMSNorm. Позиционная MLP не смешивает токены, поэтому причинность всего блока определяется маской внимания.

In [ ]:
HEADS = 2
HEAD_DIM = 2
W_1 = [[0.5, -0.2, 0.1, 0.0], [0.0, 0.3, 0.4, -0.1],
       [0.2, 0.2, -0.3, 0.5], [-0.4, 0.1, 0.0, 0.3],
       [0.1, -0.5, 0.2, 0.2], [0.3, 0.0, 0.1, -0.2]]
W_2 = [[0.2, 0.0, -0.1, 0.3, 0.0, 0.2],
       [0.0, 0.4, 0.2, -0.1, 0.1, 0.0],
       [-0.2, 0.1, 0.3, 0.0, 0.2, 0.1],
       [0.1, -0.2, 0.0, 0.4, 0.1, 0.3]]

def rmsnorm(vector, epsilon=1e-6):
    scale = 1.0 / math.sqrt(sum(value * value for value in vector) / len(vector) + epsilon)
    return [value * scale for value in vector]

def multihead_attention(sequence, causal=True):
    qs = [matvec(W_Q, state) for state in sequence]
    ks = [matvec(W_K, state) for state in sequence]
    vs = [matvec(W_V, state) for state in sequence]
    outputs = []
    for position in range(len(sequence)):
        allowed = range(position + 1) if causal else range(len(sequence))
        head_outputs = []
        for head in range(HEADS):
            start = head * HEAD_DIM
            query = qs[position][start:start + HEAD_DIM]
            head_keys = [ks[source][start:start + HEAD_DIM] for source in allowed]
            # Для values используем в точности те же разрешённые позиции.
            indices = range(position + 1) if causal else range(len(sequence))
            head_values = [vs[source][start:start + HEAD_DIM] for source in indices]
            probabilities = stable_softmax([dot(query, key) / math.sqrt(HEAD_DIM) for key in head_keys])
            head_outputs.extend(weighted_sum(probabilities, head_values))
        outputs.append(matvec(W_O, head_outputs))
    return outputs

def transformer_block(sequence, causal=True):
    attention = multihead_attention(sequence, causal=causal)
    after_attention = [rmsnorm(add(state, update)) for state, update in zip(sequence, attention)]
    outputs = []
    for state in after_attention:
        hidden = [max(0.0, value) for value in matvec(W_1, state)]
        outputs.append(rmsnorm(add(state, matvec(W_2, hidden))))
    return outputs

block_output = transformer_block(states)
assert len(block_output) == len(states)
assert all(len(row) == 4 for row in block_output)
print('shape:', (len(block_output), len(block_output[0])))

## 4. Причинный тест и намеренная поломка

Меняем только последний вход. Корректный блок сохраняет первые три выхода. Если открыть всем позициям все ключи, формы и суммы вероятностей останутся правильными, но будущее протечёт назад.

In [ ]:
mutated = [row[:] for row in states]
mutated[-1] = [9.0, -7.0, 5.0, 3.0]

causal_before = transformer_block(states, causal=True)
causal_after = transformer_block(mutated, causal=True)
for before, after in zip(causal_before[:-1], causal_after[:-1]):
    assert_close(before, after)

broken_before = transformer_block(states, causal=False)
broken_after = transformer_block(mutated, causal=False)
leaks = [max(abs(a - b) for a, b in zip(before, after))
         for before, after in zip(broken_before[:-1], broken_after[:-1])]
assert max(leaks) > 1e-3
print('максимальная утечка без маски:', round(max(leaks), 6))

## 5. Те же механизмы на torch: файлы `labs/ch04`

Длинные листинги главы лежат в отдельных файлах `labs/ch04/*.py`. Каждый файл сам проверяет себя через `assert` и печатает `OK`. Ниже мы загружаем их по явному пути через `importlib`: каждый вызов `run_lab` заново исполняет файл в собственном пространстве имён, поэтому имена `queries`, `keys`, `values` из ячеек выше не перезаписываются и скрытого состояния нет.

In [ ]:
import importlib.util
from pathlib import Path

import torch

print('torch:', torch.__version__)

def find_book_root(start=None):
    start = Path.cwd().resolve() if start is None else start
    for folder in (start, *start.parents):
        if (folder / 'labs' / 'ch04').is_dir():
            return folder
    raise FileNotFoundError('каталог labs/ch04 не найден: запустите ноутбук внутри репозитория книги')

LABS_CH04 = find_book_root() / 'labs' / 'ch04'

def run_lab(name):
    path = LABS_CH04 / f'{name}.py'
    spec = importlib.util.spec_from_file_location(f'ch04_{name}', path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)   # здесь срабатывают все assert файла
    return module

print('labs:', sorted(p.name for p in LABS_CH04.glob('*.py')))

### 5.1. Декодерный блок и маленькая языковая модель

Раздел «Собираем блок и модель» (`sec:attn-assemble`) и подраздел «Что модель пока не знает: порядок слов». Файл `labs/ch04/tiny_decoder_lm.py` собирает блок 2017 года на `torch`: причинное внимание, полносвязную ветвь, остаточный путь, LayerNorm и общую таблицу входа и выхода. Проверки: замена конца фразы не сдвигает логиты начала; один слой без позиций не различает перестановку прошлых токенов, а два слоя уже различают.

In [ ]:
tiny_lab = run_lab('tiny_decoder_lm')

print('логиты:', tuple(tiny_lab.logits_a.shape))
prefix_shift = (tiny_lab.logits_a[:, :2] - tiny_lab.logits_b[:, :2]).abs().max().item()
suffix_shift = (tiny_lab.logits_a[:, 2:] - tiny_lab.logits_b[:, 2:]).abs().max().item()
assert prefix_shift <= 1e-6 < suffix_shift
print('сдвиг логитов начала:', prefix_shift)
print('сдвиг логитов конца: ', round(suffix_shift, 6))

### 5.2. Полная модель BookLM

Раздел «Блок BookLM» (`sec:attn-booklm-block`). Файл `labs/ch04/booklm_model.py` заменяет LayerNorm на RMSNorm, GELU-ветвь на SwiGLU, общие головы на GQA и добавляет RoPE. Проверки: ровно 6 622 464 обучаемых скаляра, как в паспорте BookLM; один слой с RoPE уже различает порядок; будущее по-прежнему закрыто.

In [ ]:
booklm_lab = run_lab('booklm_model')

counts = {
    'таблица токенов': booklm_lab.booklm.tokens.weight.numel(),
    'шесть блоков': sum(p.numel() for p in booklm_lab.booklm.blocks.parameters()),
    'финальная норма': booklm_lab.booklm.norm.weight.numel(),
}
total = sum(p.numel() for p in booklm_lab.booklm.parameters())
assert sum(counts.values()) == total == 6_622_464   # выходная матрица общая с таблицей
for name, count in counts.items():
    print(f'{name:>16}: {count:>9,}'.replace(',', ' '))
print(f'{"всего":>16}: {total:>9,}'.replace(',', ' '))

### 5.3. Линейное внимание

Раздел «Линейное внимание: убираем квадрат» (`sec:attn-linear`). Файл `labs/ch04/linear_attention.py` считает внимание с $\phi(x)=\operatorname{elu}(x)+1$ двумя способами: параллельно по всем ключам и рекуррентно через состояние $S$ размера $d\times d$. Проверки: две формы совпадают, но результат заметно отличается от обычного softmax-внимания на тех же числах.

In [ ]:
linear_lab = run_lab('linear_attention')

print('расхождение параллельной и рекуррентной форм:',
      max(abs(a - b) for p, r in zip(linear_lab.parallel, linear_lab.recurrent) for a, b in zip(p, r)))
print('отличие от softmax-внимания:', round(linear_lab.gap, 6))

## Переход к BookLM

Списки qs/ks/vs соответствуют тензорам [B,H,T,Dh] с неявными B=1 и H=2. Промышленное ядро меняет размещение и порядок операций, но обязано сохранять причинный тест и оговорённый численный допуск.

**Задание.** Добавьте пакет из двух последовательностей и маску заполнителя. Докажите отдельно, что будущий токен и заполнитель не влияют на допустимые позиции.